In [11]:
import polars as pl

In [12]:
df = pl.read_csv("dataset/uicrit_public.csv" )

print(df.columns)
df.head

['rico_id', 'task', 'aesthetics_rating', 'learnability', 'efficency', 'usability_rating', 'design_quality_rating', 'comments_source', 'comments']


<bound method DataFrame.head of shape: (2_981, 9)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ rico_id ┆ task       ┆ aesthetic ┆ learnabil ┆ … ┆ usability ┆ design_qu ┆ comments_ ┆ comments  │
│ ---     ┆ ---        ┆ s_rating  ┆ ity       ┆   ┆ _rating   ┆ ality_rat ┆ source    ┆ ---       │
│ i64     ┆ str        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ing       ┆ ---       ┆ str       │
│         ┆            ┆ i64       ┆ i64       ┆   ┆ i64       ┆ ---       ┆ str       ┆           │
│         ┆            ┆           ┆           ┆   ┆           ┆ i64       ┆           ┆           │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10089   ┆ Follow     ┆ 6         ┆ 3         ┆ … ┆ 6         ┆ 6         ┆ ['human', ┆ ["Comment │
│         ┆ events to  ┆           ┆           ┆   ┆           ┆           ┆ 'human',  ┆ 1\nThe    │
│         ┆ keep track ┆           ┆     

In [6]:
df.sample(3)  # View example rows

rico_id,task,aesthetics_rating,learnability,efficency,usability_rating,design_quality_rating,comments_source,comments
i64,str,i64,i64,i64,i64,i64,str,str
9960,"""Fill in the fields to register""",6,4,4,6,6,"""['human', 'human', 'llm', 'hum…","""['LLM Comment 1\nThe expected …"
60201,"""Explore menu options.""",7,4,3,7,7,"""['human', 'human', 'both']""","""['Comment 1\nThe expected stan…"
33040,"""Shop for an item.""",7,4,4,7,7,"""['human', 'human', 'human']""","""['LLM Comment 1\nThe expected …"


In [7]:
import ast

df = df.with_columns([
    pl.col('comments').map_elements(ast.literal_eval, return_dtype=None).alias('comments'),
    pl.col('comments_source').map_elements(ast.literal_eval, return_dtype=None).alias('comments_source')
])

# Count human vs LLM vs both
from collections import Counter
comment_sources = sum(df['comments_source'].to_list(), [])
Counter(comment_sources)

<sys>:0: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
<sys>:0: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
<sys>:0: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
<sys>:0: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
<sys>:0: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
<sys>:0: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead 

Counter({'human': 8393, 'both': 1893, 'llm': 1058})

In [8]:
# check null
df.null_count()

rico_id,task,aesthetics_rating,learnability,efficency,usability_rating,design_quality_rating,comments_source,comments
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,2,0,1,0,0,0,0,0


In [11]:
# count number of values not in the range 1-10 in each column
df.select(
    [
        pl.col("aesthetics_rating").is_between(1, 10).sum().alias("aesthetics_rating"),
        pl.col("learnability").is_between(1, 5).sum().alias("learnability"),
        pl.col("efficency").is_between(1, 5).sum().alias("efficency"),
        pl.col("usability_rating").is_between(1, 10).sum().alias("usability_rating"),
        pl.col("design_quality_rating").is_between(1, 10).sum().alias("design_quality_rating"),
    ]
)

aesthetics_rating,learnability,efficency,usability_rating,design_quality_rating
u32,u32,u32,u32,u32
2981,2980,2981,2981,2981


In [20]:
sample = df[['task', 'comments', 'comments_source']].sample(5)
for comments, sources in zip(sample['comments'], sample['comments_source']):
    for comment, source in zip(comments, sources):
        print(f"[{source.upper()}] {comment}")

[[] [
['] "
[H] L
[U] L
[M] M
[A]  
[N] C
['] o
[,] m
[ ] m
['] e
[L] n
[L] t
[M]  
['] 1
[]] \
[[] [
['] '
[H] C
[U] o
[M] m
[A] m
[N] e
['] n
[,] t
[ ]  
['] 1
[H] \
[U] n
[M] T
[A] h
[N] e
[']  
[,] e
[ ] x
['] p
[H] e
[U] c
[M] t
[A] e
[N] d
[']  
[,] s
[ ] t
['] a
[B] n
[O] d
[T] a
[H] r
['] d
[]]  
[[] [
['] '
[H] C
[U] o
[M] m
[A] m
[N] e
['] n
[,] t
[ ]  
['] 1
[H] \
[U] n
[M] T
[A] h
[N] e
[']  
[,] e
[ ] x
['] p
[H] e
[U] c
[M] t
[A] e
[N] d
[']  
[,] s
[ ] t
['] a
[L] n
[L] d
[M] a
['] r
[,] d
[ ]  
['] i
[B] s
[O]  
[T] t
[H] o
[']  
[]] p
[[] [
['] '
[H] L
[U] L
[M] M
[A]  
[N] C
['] o
[,] m
[ ] m
['] e
[L] n
[L] t
[M]  
['] 1
[]] \
[[] [
['] '
[H] C
[U] o
[M] m
[A] m
[N] e
['] n
[,] t
[ ]  
['] 1
[H] \
[U] n
[M] T
[A] h
[N] e
[']  
[,] e
[ ] x
['] p
[H] e
[U] c
[M] t
[A] e
[N] d
[']  
[]] s


In [18]:
from IPython.display import display

# display a random cell from 'comments' column
display(df['comments'][1])
type(df['comments'][1])


"['Comment 1\\nThe expected standard is to make the most important information visually dominant.\\nIn the current design, the Menu option is not visually prominent.\\nTo fix this, we can enlarge the button.\\nBounding Box: [0.0, 0.04000076, 0.03710279, 0.10261065]', 'Comment 2\\nThe expected standard is that the text’s visual treatment and formatting should make it easy to read.\\nIn the current design, the background makes the foreground text difficult to read.\\nTo fix this, we can choose a different font color or choose a different contrasting background.\\nBounding Box: [0.76369902, 0.04521825, 0.99868334, 0.09391483]', 'Comment 3\\nThe expected standard is to make the most important information visually dominant.\\nIn the current design, the arrow key buttons are not visually prominent.\\nTo fix this, we can enlarge the buttons.\\nBounding Box: [0.19478963, 0.09217566, 0.82066986, 0.15857239]', 'Comment 4\\nThe expected standard is that nothing should be placed on the page arbitr

str